# UAEFA Discovery

Notebook inicial para explorar la web de UAEFA con calma, guardar respuestas y validar si merece la pena automatizar cada paso.

## Objetivo

- Revisar `robots.txt` y todos los sitemaps publicados
- Detectar URLs objetivo relacionadas con clubes, competiciones y partidos
- Guardar respuestas crudas para no repetir peticiones innecesarias
- Confirmar si la extraccion debe apoyarse en sitemap, HTML o llamadas JSON
- Mantener un ritmo de peticiones suave y facil de auditar

## 1. Preparacion del entorno

En esta seccion preparamos rutas del proyecto y cargamos el cliente reutilizable que ya incorpora pausas aleatorias y cache en disco.

In [6]:
from pathlib import Path
import re
import sys
import xml.etree.ElementTree as ET

# Localizamos la raiz del proyecto a partir de la carpeta del notebook.
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / 'src'

# Aniadimos el codigo fuente al path para poder importar modulos locales.
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from uaefa_scraper.client import GentleClient

# Directorio donde se almacenan respuestas crudas y cache local.
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Cliente con pausas suaves para no bombardear la web.
client = GentleClient(
    cache_dir=RAW_DIR / 'cache',
    min_delay=1.0,
    max_delay=2.0,
)

PROJECT_ROOT

PosixPath('/Users/macmontxinho/Desktop/Teams/Scraping UAE')

## 2. Lectura de `robots.txt`

Aqui comprobamos si existen restricciones explicitas y, sobre todo, recogemos la lista oficial de sitemaps publicados por la web.

In [7]:
# Descargamos robots.txt usando cache para no pedirlo repetidamente.
robots_url = 'https://www.uaefa.ae/robots.txt'
robots_text = client.fetch_text(robots_url)

# Mostramos el contenido para revisar reglas y sitemaps declarados.
print(robots_text)

User-Agent: *
Disallow:

sitemap: https://www.uaefa.ae/sitemap.xml
sitemap: https://www.uaefa.ae/sitemap0.xml
sitemap: https://www.uaefa.ae/sitemap1.xml
sitemap: https://www.uaefa.ae/sitemap2.xml
sitemap: https://www.uaefa.ae/sitemap3.xml
sitemap: https://www.uaefa.ae/sitemap4.xml
sitemap: https://www.uaefa.ae/sitemap5.xml
sitemap: https://www.uaefa.ae/sitemap6.xml
sitemap: https://www.uaefa.ae/sitemap7.xml
sitemap: https://www.uaefa.ae/sitemap8.xml
sitemap: https://www.uaefa.ae/sitemap9.xml
sitemap: https://www.uaefa.ae/sitemap10.xml
sitemap: https://www.uaefa.ae/sitemap11.xml
sitemap: https://www.uaefa.ae/sitemap12.xml
sitemap: https://www.uaefa.ae/sitemap13.xml
sitemap: https://www.uaefa.ae/sitemap14.xml
sitemap: https://www.uaefa.ae/sitemap15.xml
sitemap: https://www.uaefa.ae/sitemap16.xml
sitemap: https://www.uaefa.ae/sitemap17.xml
sitemap: https://www.uaefa.ae/sitemap18.xml
sitemap: https://www.uaefa.ae/sitemap19.xml
sitemap: https://www.uaefa.ae/sitemap20.xml
sitemap: https://ww

In [8]:
# Extraemos automaticamente todas las URLs de sitemap publicadas en robots.txt.
sitemap_urls = re.findall(r'(?im)^sitemap:\s*(https?://\S+)$', robots_text)

print(f'Total de sitemaps encontrados: {len(sitemap_urls)}')
sitemap_urls[:10]

Total de sitemaps encontrados: 1


['https://www.uaefa.ae/sitemap32.xml']

## 3. Utilidades para leer sitemaps

Definimos una funcion pequena para descargar y parsear cualquier sitemap XML. Asi evitamos repetir la misma logica en varias celdas.

In [9]:
def extract_urls_from_sitemap(url: str):
    """Descarga un sitemap XML y devuelve la lista de URLs publicadas."""
    text = client.fetch_text(url)
    root = ET.fromstring(text)
    ns = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}
    return [loc.text for loc in root.findall('.//sm:loc', ns)]


## 4. Primer sitemap principal

Antes de recorrer todos, inspeccionamos el sitemap principal para entender si incluye solo una parte del sitio o una fotografia completa.

In [10]:
# Leemos el sitemap principal y comprobamos cuantas URLs contiene.
main_sitemap_url = 'https://www.uaefa.ae/sitemap.xml'
main_urls = extract_urls_from_sitemap(main_sitemap_url)

len(main_urls), main_urls[:20]

(134,
 ['https://www.uaefa.ae/',
  'https://www.uaefa.ae/calendar',
  'https://www.uaefa.ae/content/UAEFA_TRANSFORMATION_STRATEGY',
  'https://www.uaefa.ae/content/playersintermediary',
  'https://www.uaefa.ae/content/regulations',
  'https://www.uaefa.ae/en',
  'https://www.uaefa.ae/en/calendar',
  'https://www.uaefa.ae/en/content/UAEFA_TRANSFORMATION_STRATEGY',
  'https://www.uaefa.ae/en/content/playersintermediary',
  'https://www.uaefa.ae/en/content/regulations',
  'https://www.uaefa.ae/en/events',
  'https://www.uaefa.ae/en/events/1',
  'https://www.uaefa.ae/en/fachannel',
  'https://www.uaefa.ae/en/footer/Beach_Soccer',
  'https://www.uaefa.ae/en/footer/Expression_of_interest',
  'https://www.uaefa.ae/en/footer/International.Relations',
  'https://www.uaefa.ae/en/footer/PlayStation_Football',
  'https://www.uaefa.ae/en/footer/PlayStation_Football_FIFA',
  'https://www.uaefa.ae/en/footer/The1stDL',
  'https://www.uaefa.ae/en/footer/ThePCup'])

## 5. Recorrido de todos los sitemaps publicados

Esta es la parte importante: recorremos todos los sitemaps encontrados en `robots.txt`, guardamos el volumen de URLs por archivo y unificamos el conjunto total.

In [11]:
# Recorremos todos los sitemaps con manejo de errores para no parar el notebook.
sitemap_summary = []
all_urls = []

for sm_url in sitemap_urls:
    try:
        urls = extract_urls_from_sitemap(sm_url)
        sitemap_summary.append({'sitemap_url': sm_url, 'url_count': len(urls), 'status': 'ok'})
        all_urls.extend(urls)
        print(f'OK  {len(urls):>5}  {sm_url}')
    except Exception as exc:
        sitemap_summary.append({'sitemap_url': sm_url, 'url_count': None, 'status': f'error: {exc}'})
        print(f'ERR        {sm_url} -> {exc}')

print('\nTotal URLs agregadas:', len(all_urls))

OK    388  https://www.uaefa.ae/sitemap32.xml

Total URLs agregadas: 388


In [12]:
# Eliminamos duplicados conservando el orden para trabajar con una lista limpia.
unique_urls = list(dict.fromkeys(all_urls))

print('Total URLs unicas:', len(unique_urls))
unique_urls[:20]

Total URLs unicas: 388


['https://www.uaefa.ae/',
 'https://www.uaefa.ae/club/1',
 'https://www.uaefa.ae/club/11',
 'https://www.uaefa.ae/club/12',
 'https://www.uaefa.ae/club/13',
 'https://www.uaefa.ae/club/14',
 'https://www.uaefa.ae/club/15',
 'https://www.uaefa.ae/club/16',
 'https://www.uaefa.ae/club/17',
 'https://www.uaefa.ae/club/18',
 'https://www.uaefa.ae/club/19',
 'https://www.uaefa.ae/club/2',
 'https://www.uaefa.ae/club/20',
 'https://www.uaefa.ae/club/21',
 'https://www.uaefa.ae/club/22',
 'https://www.uaefa.ae/club/23',
 'https://www.uaefa.ae/club/24',
 'https://www.uaefa.ae/club/25',
 'https://www.uaefa.ae/club/26',
 'https://www.uaefa.ae/club/27']

## 6. Filtrado por tipos de URL

Ahora separamos las URLs que nos interesan por categoria. Esto nos dira rapidamente si los clubes y las competiciones estan realmente publicados en sitemap o si habra que ir a red/API.

In [13]:
# Filtramos varias familias de paginas utiles para analisis posterior.
clubs_pages = [u for u in unique_urls if u.endswith('/en/clubs')]
club_urls = [u for u in unique_urls if '/en/club/' in u]
competitions_pages = [u for u in unique_urls if u.endswith('/en/competitions')]
competition_urls = [u for u in unique_urls if '/en/competition/' in u]
matches_pages = [u for u in unique_urls if u.endswith('/en/matches')]
match_urls = [u for u in unique_urls if '/en/match/' in u]

print('clubs page:', clubs_pages[:5])
print('club urls:', len(club_urls))
print('competitions page:', competitions_pages[:5])
print('competition urls:', len(competition_urls))
print('matches page:', matches_pages[:5])
print('match urls:', len(match_urls))

clubs page: ['https://www.uaefa.ae/en/clubs']
club urls: 57
competitions page: ['https://www.uaefa.ae/en/competitions']
competition urls: 0
matches page: ['https://www.uaefa.ae/en/matches']
match urls: 0


In [14]:
# Muestra rapida de cada categoria para verificar formato y patrones.
club_urls[:10], competition_urls[:10], match_urls[:10]

(['https://www.uaefa.ae/en/club/1',
  'https://www.uaefa.ae/en/club/11',
  'https://www.uaefa.ae/en/club/12',
  'https://www.uaefa.ae/en/club/13',
  'https://www.uaefa.ae/en/club/14',
  'https://www.uaefa.ae/en/club/15',
  'https://www.uaefa.ae/en/club/16',
  'https://www.uaefa.ae/en/club/17',
  'https://www.uaefa.ae/en/club/18',
  'https://www.uaefa.ae/en/club/19'],
 [],
 [])

## 7. Inspeccion de una pagina concreta

Si una ficha devuelve solo el HTML base de Angular, sabremos que los datos reales no vienen renderizados en servidor y tendremos que buscar llamadas internas.

In [15]:
# Probamos una URL representativa para ver si devuelve datos o solo la app base.
sample_url = 'https://www.uaefa.ae/en/club/1'
sample_html = client.fetch_text(sample_url)

print(sample_html[:1500])

<!doctype html>
<html>

<head>
    <base href='/'>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8">
    <meta http-equiv="Content-Security-Policy"
        content="script-src 'self' 'unsafe-inline' 'unsafe-eval' https://www.googletagmanager.com/gtag/js https://www.google-analytics.com/analytics.js https://platform.twitter.com/widgets.js https://api.instagram.com/v1/users/self/media/recent/;">
    <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no" />
    <meta name="description" content="UAE Football Association" />
    <meta name="keyword" content="UAE, UAEFA, UAEFOOTBALL, uae sports, uae football" />
    <meta content="text/html; charset=UTF-8; X-Content-Type-Options=nosniff" http-equiv="Content-Type" />
    <meta name="author" content="designfort" />
    <meta name="google-site-verification" content="bKVDv0IREyQiQcjNkUsA2FhaGZohebql6jxpqXiezjE" />
    <!-- Global site tag (gtag.js) - Google Analytics -->
    <script async src=

In [16]:
# Pequenas comprobaciones para detectar si estamos ante una SPA tipo Angular.
print('Contiene <app-root>?', '<app-root>' in sample_html)
print('Contiene main.js?', 'main.' in sample_html)
print('Contiene datos visibles de club?', 'club' in sample_html.lower())

Contiene <app-root>? True
Contiene main.js? True
Contiene datos visibles de club? False


## 8. Diagnostico rapido

Esta celda resume la situacion actual para decidir el siguiente movimiento sin tener que reinterpretar manualmente toda la salida anterior.

In [17]:
# Resumen ejecutivo del descubrimiento inicial.
summary = {
    'total_sitemaps': len(sitemap_urls),
    'total_unique_urls': len(unique_urls),
    'club_urls_found': len(club_urls),
    'competition_urls_found': len(competition_urls),
    'match_urls_found': len(match_urls),
    'sample_page_looks_like_spa': '<app-root>' in sample_html,
}

summary

{'total_sitemaps': 1,
 'total_unique_urls': 388,
 'club_urls_found': 57,
 'competition_urls_found': 0,
 'match_urls_found': 0,
 'sample_page_looks_like_spa': True}

## 9. Siguiente paso sugerido

Si los sitemaps ya exponen muchas fichas, podemos seguir por esa ruta. Si no, el siguiente notebook deberia centrarse en detectar y validar llamadas JSON concretas hechas por la web en el navegador.